# Detecció d'opinions (pràctica 2.a)
### Recursos

- Movie Reviews Corpus

### Enunciat

- Implementeu un detector d'opinions positives o negatives amb alguns algoritmes d'aprenentatge supervisat de l'sklearn # perceptro, svm i arbres (3 models diferents)

- Utilitzeu com a dades el Movie Reviews Corpus de l'NLTK

- Dissenyeu i apliqueu un protocol de validació

- Utilitzeu el preprocés que cregueu més convenient: eliminació d'stop words, signes de puntuació... (reduir la dimensionalitat del vocabulari (amb lemes de diccionary o symsets))

- Utilitzeu el CountVectorizer per representar la informació

- Doneu la precisió (accuracy) i les matrius de confusió

- Analitzeu els resultats (Analitzar les frases que fallen)

num_features = 5000 <-- fixar mida de diccionari

## NLTK’s Movie Reviews Corpus

### Polarity corpus:

- 1000 exemples positius i 1000 negatius

### Requeriments:

In [2]:
import re
import nltk
nltk.download('movie_reviews')
from nltk.corpus import movie_reviews as mr

[nltk_data] Downloading package movie_reviews to
[nltk_data]     C:\Users\david\AppData\Roaming\nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!


### Ús:

In [3]:
mr.fileids('pos')[:2]

['pos/cv000_29590.txt', 'pos/cv001_18431.txt']

In [4]:
len(mr.fileids('neg'))

1000

In [5]:
mr.words()

['plot', ':', 'two', 'teen', 'couples', 'go', 'to', ...]

In [6]:
mr.raw(mr.fileids('pos')[:2][0]).split("\n")[:10]

["films adapted from comic books have had plenty of success , whether they're about superheroes ( batman , superman , spawn ) , or geared toward kids ( casper ) or the arthouse crowd ( ghost world ) , but there's never really been a comic book like from hell before . ",
 "for starters , it was created by alan moore ( and eddie campbell ) , who brought the medium to a whole new level in the mid '80s with a 12-part series called the watchmen . ",
 'to say moore and campbell thoroughly researched the subject of jack the ripper would be like saying michael jackson is starting to look a little odd . ',
 'the book ( or " graphic novel , " if you will ) is over 500 pages long and includes nearly 30 more that consist of nothing but footnotes . ',
 "in other words , don't dismiss this film because of its source . ",
 "if you can get past the whole comic book thing , you might find another stumbling block in from hell's directors , albert and allen hughes . ",
 "getting the hughes brothers to di

## CountVectorizer de l'sklearn

Codificador bag of words (TF-IDF)

### Exemple:

This is the first document.
This document is the second document.
And this is the third one.
Is this the first document?

### Matriu resultant:

0 1 1 1 0 0 1 0 1
0 2 0 1 0 1 1 0 1
1 0 0 1 1 0 1 1 1
0 1 1 1 0 0 1 0 1

### Diccionari:

index	word  
0:      and  
1:    document  
2:     first  
3:      is  
4:     one  
5:    second  
6:     the  
7:    third  
8:    this

### Referència:

https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html

# Detecció d'opinions (pràctica 2.b)

### Enunciat

- Implementeu un detector d'opinions positives o negatives no supervisat

    1. Apliqueu l'UKB per obtenir els synsets de les paraules (nltk lesk)
    2. Obtingueu els valors SentiWordnet de cada synset

- Utilitzeu com a dades el/els conjunts de test que hagueu utilitzat a la pràctica 2.a

- Penseu en com podeu combinar aquests valors per obtenir un resultat

- Penseu que fareu si el synset no hi és a SentiWordnet

- Penseu quines categories utilitzareu:

    - només adjectius
    - noms, adjectius i adverbis
    - noms, adjectius, verbs i adverbis (com a minim fer aquestes combinacions)

- Analitzeu els resultats i compareu-los amb els de la part supervisada

    - Desambiguació de 10 frases amb lesk and UKB i comparar la desambiguació que han fet -----> (Com que la diferencia no es tan gran farem servir lesk)

svm, perceptro, preprocessat | noms, adjectius i adverbis i pensar només adjectius

In [36]:
# UKB
from textserver import TextServer
# Lesk
from nltk.corpus import wordnet as wn
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')
wnl = nltk.stem.WordNetLemmatizer()
def lemmatize(p):
  d = {'NN': 'n', 'NNS': 'n', 
       'JJ': 'a', 'JJR': 'a', 'JJS': 'a', 
       'VB': 'v', 'VBD': 'v', 'VBG': 'v', 'VBN': 'v', 'VBP': 'v', 'VBZ': 'v', 
       'RB': 'r', 'RBR': 'r', 'RBS': 'r'}
  if p[1] in d:
    return wnl.lemmatize(p[0], pos=d[p[1]]), d[p[1]]
  return p[0], None

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\david\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\david\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\david\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


In [8]:
frase = mr.raw(mr.fileids('pos')[:2][0]).split("\n")[0]
words = frase.split()
tags = nltk.pos_tag(words)
resultat = []
print(frase)
for pair in tags:
    lem = lemmatize(pair)
    if lem[1] != None:
        synset = nltk.wsd.lesk(frase, *lem)
        if synset:
            print(synset.name(), synset.definition())

films adapted from comic books have had plenty of success , whether they're about superheroes ( batman , superman , spawn ) , or geared toward kids ( casper ) or the arthouse crowd ( ghost world ) , but there's never really been a comic book like from hell before . 
movie.n.01 a form of entertainment that enacts a story by sound and a sequence of images giving the illusion of continuous movement
adapt.v.01 make fit for, or change to suit a new purpose
comic.a.02 of or relating to or characteristic of comedy
script.n.01 a written version of a play or other dramatic composition; used in preparing for a performance
induce.v.02 cause to do; cause to act in a specified manner
induce.v.02 cause to do; cause to act in a specified manner
plenty.n.01 a full supply
success.n.03 a state of prosperity or fame
batman.n.01 an orderly assigned to serve a British military officer
demigod.n.01 a person with great powers and abilities
spawn.n.01 the mass of eggs deposited by fish or amphibians or mollus

In [38]:
frase = mr.raw(mr.fileids('pos')[:2][0]).split("\n")[0]
print(frase)
ts = TextServer('davidvazquez', 'c4tSX16B!', 'senses') 
synset_table = ts.senses(frase)

films adapted from comic books have had plenty of success , whether they're about superheroes ( batman , superman , spawn ) , or geared toward kids ( casper ) or the arthouse crowd ( ghost world ) , but there's never really been a comic book like from hell before . 


In [39]:
for frase_synset in synset_table:
    for synset in frase_synset:
        if synset[4] != 'N/A':
            syn = wn.synset_from_pos_and_offset(synset[4][-1], int(synset[4][:-2]))
            print(synset[0], syn.name(), syn.definition())
    print()

films movie.n.01 a form of entertainment that enacts a story by sound and a sequence of images giving the illusion of continuous movement
batman batman.n.01 an orderly assigned to serve a British military officer
or gold.n.03 a soft yellow malleable ductile (trivalent and univalent) metallic element; occurs mainly as nuggets in rocks and alluvial deposits; does not react with most chemicals but is attacked by chlorine and aqua regia
or gold.n.03 a soft yellow malleable ductile (trivalent and univalent) metallic element; occurs mainly as nuggets in rocks and alluvial deposits; does not react with most chemicals but is attacked by chlorine and aqua regia



In [11]:
deu_frases = mr.raw(mr.fileids('pos')[:2][0]).split("\n")[:10]
for frase in deu_frases:
    tags = nltk.pos_tag(frase)
    synset = nltk.wsd.lesk(frase, 'bank', 'n')
    synset.name(), synset.definition()

TypeError: tokens: expected a list of strings, got a string